# 09 · Naive Bayes and Random Forest (three separate submissions)

Builds the three files you asked for: Naive Bayes alone, Random Forest alone, and an ensemble of the two.

**Needs:** `features2.parquet` (02). **Writes:** `nb_*`, `rf_*`, `submission_naive_bayes.csv`, `submission_random_forest.csv`, `submission_nb_rf_ensemble.csv`. **Time:** about 12 minutes.

Naive Bayes assumes every clue is independent, which is false here (many creatinine summaries say the same thing). So it only sees 50 features, chosen *inside each training fold* by predictive strength while skipping any feature highly correlated with one already picked.
Out-of-fold AUC: Naive Bayes **0.722**, Random Forest **0.730**, ensemble **0.730**. Both are clearly below the boosted-tree models (about 0.753).

**Setup.** Install the libraries once (uncomment), then set the two paths in the next cell.

In [ ]:
# %pip install -q lightgbm xgboost scikit-learn pandas pyarrow scipy

In [ ]:
import os, sys, json, time, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

# ---- EDIT THESE TWO PATHS --------------------------------------------------
DATA = os.environ.get('KAGGLE_DATA', 'data')   # folder holding the unzipped competition CSVs (train.csv, test.csv, patients.csv, ...)
WORK = os.environ.get('KAGGLE_WORK', 'work')   # folder where features, predictions and submissions are saved (shared by all notebooks)
# -----------------------------------------------------------------------------
Path(WORK).mkdir(parents=True, exist_ok=True)
print('DATA ->', Path(DATA).resolve()); print('WORK ->', Path(WORK).resolve())

In [ ]:
import pandas as pd, numpy as np, sys, time
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import QuantileTransformer

def main(which):
    D=DATA; W=WORK
    drop=("hrs_","visits_","n_transfers","first_unit","last_unit","icu_","had_ed","ward_after","proc_","n_proc")
    X=pd.read_parquet(f'{W}/features2.parquet'); tr=pd.read_csv(f'{D}/train.csv'); te=pd.read_csv(f'{D}/test.csv')
    T=X.merge(tr,on='encounter_id'); S=te.merge(X,on='encounter_id',how='left'); y=T.readmit_30d.values
    feats=[c for c in T.columns if c not in('encounter_id','readmit_30d') and not c.startswith(drop)]
    cat=[c for c in feats if not pd.api.types.is_numeric_dtype(T[c])]; num=[c for c in feats if c not in cat and T[c].notna().sum()>0 and T[c].nunique()>1]
    folds=list(StratifiedKFold(5,shuffle=True,random_state=0).split(T,y)); t0=time.time()
    oof=np.zeros(len(T)); tst=np.zeros(len(S))
    if which=='rf':
        from sklearn.ensemble import RandomForestClassifier
        def enc(df):
            d=df[num].copy()
            for c in cat: d[c]=pd.Categorical(df[c],categories=sorted(T[c].dropna().unique())).codes.astype(float)
            return d.fillna(-999).values.astype(np.float32)
        Zt,Zs=enc(T),enc(S); print('RF matrix',Zt.shape,flush=True)
        for f,(a,b) in enumerate(folds):
            m=RandomForestClassifier(250,min_samples_leaf=50,max_features=0.1,max_samples=0.5,n_jobs=2,random_state=f).fit(Zt[a],y[a])
            oof[b]=m.predict_proba(Zt[b])[:,1]; tst+=m.predict_proba(Zs)[:,1]/5
            print('rf fold',f,'%.5f'%roc_auc_score(y[b],oof[b]),'t=%.0f'%(time.time()-t0),flush=True)
        print('RF OOF AUC %.5f'%roc_auc_score(y,oof)); np.save(f'{W}/rf_oof.npy',oof); np.save(f'{W}/rf_test.npy',tst)
    else:
        from sklearn.naive_bayes import GaussianNB
        A=pd.concat([T[num],S[num]]); lo,hi=T[num].quantile(.005),T[num].quantile(.995); med=T[num].clip(lo,hi,axis=1).median()
        def prep(df):
            N=df[num].clip(lo,hi,axis=1).fillna(med); M=df[[c for c in num if T[c].isna().mean()>0.01]].isna().astype(float).add_suffix('_na')
            return pd.concat([N,M,df[cat].astype(object).fillna('NA')],axis=1)
        Z=pd.get_dummies(pd.concat([prep(T),prep(S)],ignore_index=True),columns=cat,dtype=float)
        Z=Z.loc[:,Z.nunique()>1].astype(np.float32); cols=list(Z.columns); Zt,Zs=Z.iloc[:len(T)].values,Z.iloc[len(T):].values; print('NB pool',Zt.shape,flush=True)
        KS=[10,20,30,50,80]; res={k:np.zeros(len(T)) for k in KS}; tsts={k:np.zeros(len(S)) for k in KS}
        for f,(a,b) in enumerate(folds):
            qt=QuantileTransformer(n_quantiles=200,output_distribution='normal',subsample=60000,random_state=0).fit(Zt[a])
            Qa,Qb,Qs=qt.transform(Zt[a]),qt.transform(Zt[b]),qt.transform(Zs)
            auc=np.array([abs(roc_auc_score(y[a],Qa[:,j])-.5) if Qa[:,j].std()>0 else 0 for j in range(Qa.shape[1])])   # selection uses training part only
            order=np.argsort(-auc); sub=Qa[np.random.RandomState(0).choice(len(a),15000,replace=False)]; chosen=[]
            for j in order[:300]:
                if all(abs(np.corrcoef(sub[:,j],sub[:,i])[0,1])<0.6 for i in chosen): chosen.append(j)
                if len(chosen)>=max(KS): break
            for k in KS:
                sel=chosen[:k]; m=GaussianNB(var_smoothing=1e-3).fit(Qa[:,sel],y[a])
                res[k][b]=m.predict_proba(Qb[:,sel])[:,1]; tsts[k]+=m.predict_proba(Qs[:,sel])[:,1]/5
            print('nb fold',f,{k:round(roc_auc_score(y[b],res[k][b]),4) for k in KS},'t=%.0f'%(time.time()-t0),flush=True)
            if f==0: print('top features fold0:',[cols[j] for j in chosen[:12]],flush=True)
        sc={k:roc_auc_score(y,res[k]) for k in KS}; print('NB OOF AUC by k:',{k:round(v,5) for k,v in sc.items()})
        kb=max(sc,key=sc.get); print('best k',kb); np.save(f'{W}/nb_oof.npy',res[kb]); np.save(f'{W}/nb_test.npy',tsts[kb])

In [ ]:
main('rf'); main('nb')

### 9.1 Calibrate and write the three files

In [ ]:
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
W=WORK
X=pd.read_parquet(f'{W}/features2.parquet',columns=['encounter_id']); tr=pd.read_csv(f'{DATA}/train.csv'); te=pd.read_csv(f'{DATA}/test.csv'); ss=pd.read_csv(f'{DATA}/sample_submission.csv')
y=X.merge(tr,on='encounter_id').readmit_30d.values
lg=lambda p:np.log(np.clip(p,1e-12,1-1e-12)/(1-np.clip(p,1e-12,1-1e-12)))
nb_o,nb_t=lg(np.load(f'{W}/nb_oof.npy')),lg(np.load(f'{W}/nb_test.npy')); rf_o,rf_t=lg(np.load(f'{W}/rf_oof.npy')),lg(np.load(f'{W}/rf_test.npy'))
print('raw OOF AUC  NB %.5f  RF %.5f  | corr of logits %.3f'%(roc_auc_score(y,nb_o),roc_auc_score(y,rf_o),np.corrcoef(nb_o,rf_o)[0,1]))
# ensemble weight: logistic stacker on [NB, RF] logits, evaluated with nested CV
mu_n,sd_n,mu_r,sd_r=nb_o.mean(),nb_o.std(),rf_o.mean(),rf_o.std()
O=np.column_stack([(nb_o-mu_n)/sd_n,(rf_o-mu_r)/sd_r]); Ts=np.column_stack([(nb_t-mu_n)/sd_n,(rf_t-mu_r)/sd_r]); nest=np.zeros(len(y))
for a,b in StratifiedKFold(5,shuffle=True,random_state=42).split(O,y): nest[b]=LogisticRegression(C=1).fit(O[a],y[a]).decision_function(O[b])
print('Ensemble (NB+RF stacked) nested-CV OOF AUC %.5f'%roc_auc_score(y,nest))
for w in [0.2,0.3,0.4,0.5]:
    z=(w*(nb_o-nb_o.mean())/nb_o.std()+(1-w)*(rf_o-rf_o.mean())/rf_o.std()); print('  simple weight NB=%.1f  AUC %.5f'%(w,roc_auc_score(y,z)))
st=LogisticRegression(C=1).fit(O,y); print('stacker coefs (NB, RF):',st.coef_.round(3))
def platt(o,t): m=LogisticRegression(C=1e6).fit(o.reshape(-1,1),y); return m.predict_proba(t.reshape(-1,1))[:,1]   # monotone: keeps ranking, makes real probabilities
outs={'submission_naive_bayes.csv':platt(nb_o,nb_t),'submission_random_forest.csv':platt(rf_o,rf_t),'submission_nb_rf_ensemble.csv':st.predict_proba(Ts)[:,1]}
for f,p in outs.items():
    s=pd.DataFrame({'encounter_id':te.encounter_id,'readmit_30d':np.round(p,6)}); s.to_csv(f'{W}/{f}',index=False)
    print(f, s.shape, 'ids match sample:',s.encounter_id.tolist()==ss.encounter_id.tolist(),'NaN',s.readmit_30d.isna().sum(),'range %.4f-%.4f'%(p.min(),p.max()),'mean %.4f'%p.mean())

In [ ]:
import pandas as pd
def validate_submission(path):
    """Checks a file against sample_submission.csv: same ids in the same order, no gaps, probabilities in [0, 1]."""
    s = pd.read_csv(path); ss = pd.read_csv(f'{DATA}/sample_submission.csv')
    ok = s.encounter_id.tolist() == ss.encounter_id.tolist()
    print(Path(path).name, '| rows', len(s), '| ids match sample:', ok, '| NaN:', int(s.readmit_30d.isna().sum()),
          '| all in [0,1]:', bool(s.readmit_30d.between(0, 1).all()), '| mean %.4f' % s.readmit_30d.mean())
    assert ok and s.readmit_30d.notna().all() and s.readmit_30d.between(0, 1).all()

In [ ]:
for f in ['submission_naive_bayes.csv', 'submission_random_forest.csv', 'submission_nb_rf_ensemble.csv']:
    validate_submission(f'{WORK}/{f}')